In [ ]:
import h5py
import pandas as pd
from PIL import Image
import os
import cv2
import numpy as np
from datetime import datetime, timedelta
import re
from tqdm import tqdm
from pdf2image import convert_from_path
import matplotlib.pyplot as plt
from PIL import Image

In [ ]:
# import os
# from pdf2image import convert_from_path
# from PIL import Image
#
# directory = "D:/Workspace_1/TEMPO/dataset/m1_image"
#
# for root, dirs, files in os.walk(directory):
#     for file in files:
#         if file.lower().endswith('.pdf'):
#             pdf_path = os.path.join(root, file)
#
#             try:
#                 images = convert_from_path(pdf_path)
#                 pdf_name = os.path.splitext(file)[0]
#
#                 for i, image in enumerate(images):

#                     width, height = image.size
#                     cropped_image = image.crop((0, 0, width, height // 2))
#

#                     png_path = os.path.join(root, f"{pdf_name}_page_{i+1:03d}.png")
#                     cropped_image.save(png_path, 'PNG')
#
#             except Exception as e:
#                 print(f"process {file} error: {e}")

In [ ]:
def image_pro_mic(c_date, days_list, img_type, size, pt_id, pt_name):
    images= []

    if os.path.exists(f"{source_path}/{pt_id}{pt_name}/{c_date}/{img_type}/"):
        pic_list = os.listdir(f"{source_path}/{pt_id}{pt_name}/{c_date}/{img_type}/")
        for pic in pic_list:
            if pic.endswith(".jpg") or pic.endswith(".png"):
                img = np.array(Image.open(f"{source_path}/{pt_id}{pt_name}/{c_date}/{img_type}/{pic}").convert('L'))
                if img is not None:
                    resized = cv2.resize(img, (size, size))
                    img_normalized = (resized - resized.min()) / (resized.max() - resized.min() + 1e-8)
                    images.append(img_normalized)

    else:
        for cls_day in days_list:
            target_path = f"{source_path}/{pt_id}{pt_name}/{cls_day}/{img_type}/"
            if os.path.exists(target_path):
                pic_list = os.listdir(target_path)
                for pic in pic_list:
                    if pic.endswith(".jpg") or pic.endswith(".png"):
                        img = np.array(Image.open(f"{target_path}{pic}").convert('L'))
                        if img is not None:
                            resized = cv2.resize(img, (size, size))
                            img_normalized = (resized - resized.min()) / (resized.max() - resized.min() + 1e-8)
                            images.append(img_normalized)

    return images

In [ ]:
def image_pro(c_date, days_list, img_type, size, pt_id, pt_name):
    images= []

    if os.path.exists(f"{source_path}/{pt_id}{pt_name}/{c_date}/{img_type}/"):
        pic_list = os.listdir(f"{source_path}/{pt_id}{pt_name}/{c_date}/{img_type}/")
        for pic in pic_list:
            if pic.endswith(".jpg") or pic.endswith(".png"):
                img = np.array(Image.open(f"{source_path}/{pt_id}{pt_name}/{c_date}/{img_type}/{pic}").convert('L'))
                if img is not None:
                    resized = cv2.resize(img, (size, size))
                    img_normalized = (resized - resized.min()) / (resized.max() - resized.min() + 1e-8)
                    bottom_ratio = 0.1
                    right_ratio = 0.1

                    h, w = img_normalized.shape
                    bottom_mask = int(h * bottom_ratio)
                    right_mask = int(w * right_ratio)

                    img_normalized[-bottom_mask:, :] = 0
                    img_normalized[:, -right_mask:] = 0
                    images.append(img_normalized)

    else:
        for cls_day in days_list:
            target_path = f"{source_path}/{pt_id}{pt_name}/{cls_day}/{img_type}/"
            if os.path.exists(target_path):
                pic_list = os.listdir(target_path)
                for pic in pic_list:
                    if pic.endswith(".jpg") or pic.endswith(".png"):
                        img = np.array(Image.open(f"{target_path}{pic}").convert('L'))
                        if img is not None:
                            resized = cv2.resize(img, (size, size))
                            img_normalized = (resized - resized.min()) / (resized.max() - resized.min() + 1e-8)
                            bottom_ratio = 0.1
                            right_ratio = 0.1

                            h, w = img_normalized.shape
                            bottom_mask = int(h * bottom_ratio)
                            right_mask = int(w * right_ratio)


                            img_normalized[-bottom_mask:, :] = 0
                            img_normalized[:, -right_mask:] = 0

                            images.append(img_normalized)

    return images


In [ ]:
data = pd.read_csv("D:/Workspace_1/TEMPO/dataset/tabular_input.csv")
source_path = "D:/Workspace_1/TEMPO/dataset/m1_image"

mk_list = os.listdir(source_path)
name_dict = {}
for item in mk_list:
    match = re.match(r'(\d+)(.*)', item)
    if match:
        number_part = match.group(1)
        name_part = match.group(2)
        name_dict[int(number_part)] = name_part

In [ ]:
patient_id = data.id.tolist()
collection_date = data.collection_date.tolist()

In [ ]:
with h5py.File("img_dataset.h5", 'a') as f:
    for idx in tqdm(range(len(patient_id))):
        pt_id = patient_id[idx]
        c_date = str(pd.to_datetime(collection_date[idx])).split(" ")[0]
        base_date = datetime.strptime(c_date, '%Y-%m-%d').date()

        days_list = []
        for i in range(-7, 1):  # -7到0，共8天（包含当天）
            target_date = base_date + timedelta(days=i)
            days_list.append(target_date.strftime('%Y%m%d'))
        c_date = c_date.replace('-', '')

        # 先搜索是否有c_date
        MRI_AXI_dir = f"{source_path}/{pt_id}{name_dict[pt_id]}/{c_date}/MRI_AXI_T1/"
        MRI_COR_dir = f"{source_path}/{pt_id}{name_dict[pt_id]}/{c_date}/MRI_COR_T1/"
        MRI_SAG_dir = f"{source_path}/{pt_id}{name_dict[pt_id]}/{c_date}/MRI_SAG_T1/"
        MIC_dir = f"{source_path}/{pt_id}{name_dict[pt_id]}/{c_date}/检验科显微镜/"

        # image collections
        axi_img = image_pro(c_date, days_list, "MRI_AXI_T1", 256, pt_id, name_dict[pt_id])
        cor_img = image_pro(c_date, days_list, "MRI_COR_T1", 256, pt_id, name_dict[pt_id])
        sag_img = image_pro(c_date, days_list, "MRI_SAG_T1", 256, pt_id, name_dict[pt_id])
        mic_img = image_pro_mic(c_date, days_list, "检验科显微镜", 256, pt_id, name_dict[pt_id])

        data_dict = {
            'axi_img': axi_img,
            'cor_img': cor_img,
            'sag_img': sag_img,
            'mic_img': mic_img
        }
        group_path = f"{pt_id}/{c_date}"
        group = f.create_group(group_path)
        for img_type, img_list in data_dict.items():
            if img_list:
                type_group = group.create_group(img_type)
                for i, img in enumerate(img_list):
                    type_group.create_dataset(
                        f"img_{i:04d}",
                        data=img.astype(np.float32),
                        compression="gzip"
                    )


In [ ]:
with h5py.File("img_dataset.h5", 'r') as f:
    print(f['106']['20241223'].keys())
    sample = f["106"]['20241223']['mic_img']['img_0000'][:]

    # plt.imshow(sample, cmap="gray")